In [1]:
import os


In [2]:
os.chdir("../")
%pwd

'/Users/prasad/Learning/Projects/Kidney-Disease-Classification-Project'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    kaggle_dataset: str
    unzip_dir: Path


In [4]:
from cnnClassifierKidneyDisease.constants import *
from cnnClassifierKidneyDisease.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path: Path = CONFIG_FILE_PATH,
        params_file_path: Path = PARAMS_FILE_PATH) -> None:
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        
        create_directories([self.config.artifacts_root])



    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            kaggle_dataset=config.kaggle_dataset,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

In [10]:
pip install kaggle

Note: you may need to restart the kernel to use updated packages.


In [20]:
import os
import kagglehub
import shutil
from cnnClassifierKidneyDisease.logging import logger
from cnnClassifierKidneyDisease.utils.common import get_size

In [23]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
    
    def download_kaggle_data(self) -> str:
        '''
        Fetches the data from Kaggle using the provided URL in the configuration and saves it to the specified local path.
        
        returns:
            str: The path where the data has been downloaded.
        '''
        try:
            os.makedirs(os.path.dirname(self.config.unzip_dir), exist_ok=True )
            os.environ["KAGGLEHUB_CACHE"] = self.config.unzip_dir

            logger.info(f"Starting download of data from Kaggle: {self.config.kaggle_dataset}")  

            cache_path = kagglehub.dataset_download(self.config.kaggle_dataset)
            
            logger.info(f"Data downloaded to: {cache_path}")
            return cache_path
        except Exception as e:
            logger.error(f"Error occurred while downloading data from Kaggle: {self.config.kaggle_dataset}")
            raise e

In [24]:
try:
    config_manager = ConfigurationManager()
    data_ingestion_config = config_manager.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    downloaded_path = data_ingestion.download_kaggle_data()
    logger.info(f"Size of downloaded data: {get_size(downloaded_path)}")
except Exception as e:
    logger.error(f"An error occurred during data ingestion: {str(e)}")

[2026-05-09 14:28:23,276: INFO: common: YAML file loaded successfully: config/config.yaml]
[2026-05-09 14:28:23,277: INFO: common: YAML file loaded successfully: params.yaml]
[2026-05-09 14:28:23,278: INFO: common: Directory created successfully: artifacts]
[2026-05-09 14:28:23,278: INFO: common: Directory created successfully: artifacts/data_ingestion]
[2026-05-09 14:28:23,279: INFO: 3469315637: Starting download of data from Kaggle: nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone]


100%|██████████| 1.52G/1.52G [09:13<00:00, 2.94MB/s]

Extracting model files...


[2026-05-09 14:37:44,717: INFO: 3469315637: Data downloaded to: artifacts/data_ingestion/ct-kidney-dataset/datasets/nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone/versions/1]
[2026-05-09 14:37:44,719: ERROR: 2845515996: An error occurred during data ingestion: Argument path of type <class 'str'> to <function get_size at 0x1218c2040> does not match annotation type <class 'pathlib.Path'>]
